<a href="https://colab.research.google.com/github/JaudatUllahKhan/my-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JaudatUllahKhan/my-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

# Week 7: Content Action Playbook & Research Paper Exports (Lane 1: Content Refresh & Decay)

## 1. Ranked Actions & Reason Codes
This playbook turns out-of-fold risk scores from our Week 6 validation pipeline into an actionable, human-in-the-loop prioritization queue for SEO and editorial teams.

### Archetype to Action Mapping
* **High Staleness + High Drop Risk (`REFRESH_CONTENT`):** Update statistics, replace outdated links, and refresh stale copy.
* **Low CTR + Stable Position (`OPTIMIZE_METADATA`):** Rewrite title tags, meta descriptions, and featured snippet formatting.
* **High Traffic Loss + Young Content (`AUDIT_TECHNICAL_LINKS`):** Check canonical tags, indexing issues, keyword cannibalization, and boost internal links.
* **Stable Performance (`MONITOR`):** No immediate intervention required.

---

## 2. Intended Use and Limits
* **Who Uses This:** SEO Specialists, Content Strategists, and Editorial leads.
* **Intended Purpose:** Prioritization decision-support tool to queue pages needing manual inspection and refresh.
* **Non-Automated Limits (The No-Go List):**
  1. **No Automated Content Rewriting:** Generative models or automated templates must never directly push copy edits to production without editorial sign-off.
  2. **No Automated URL Redirects/Deletions:** Structural changes, canonical overrides, or page deletions require manual SEO impact review.
  3. **Brand-Sensitive / Legal Pages:** Compliance, terms of service, and core conversion pages are strictly excluded from automated intervention queues.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [1]:
# 2. Ranked Action Queue Generation with Reason Codes & Exports
import os, sys, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

# ---------------------------------------------------------
# Step A: Setup & Data Loading
# ---------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

# Create output directories
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Dynamic ID column
possible_id_cols = ["page_id", "url", "page", "path", "id"]
page_col = next((col for col in possible_id_cols if col in df.columns), df.columns[0])

# Age group setup for GroupKFold
df["age_group"] = pd.qcut(df["content_age_days"], q=5, labels=False, duplicates="drop")
group_col = "age_group"

# Label assignment
if "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
else:
    df["is_declining_label"] = (df.get("traffic_change_pct", 0) < 0).astype(int)

feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "ctr", "avg_position"]
X = df[feature_cols].fillna(0)
y = df["is_declining_label"]

# Train model on grouped split
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=df[group_col]))

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

# Generate decay probability score
df["decay_probability"] = rf.predict_proba(X)[:, 1]

# ---------------------------------------------------------
# Step B: Reason Code Assignment Logic
# ---------------------------------------------------------
def assign_action_playbook(row):
    prob = row["decay_probability"]
    age = row["content_age_days"]
    days_updated = row["days_since_last_update"]
    ctr = row["ctr"]

    if prob >= 0.65 and (days_updated > 365 or age > 365):
        return (
            "REFRESH_CONTENT",
            "HIGH_PRIORITY",
            "RC_STALE_DECAY: Content age >365d with high predicted decay risk."
        )
    elif prob >= 0.50 and ctr < 0.02:
        return (
            "OPTIMIZE_METADATA",
            "MEDIUM_PRIORITY",
            "RC_LOW_CTR: Below-average CTR on ranking keywords; needs snippet refresh."
        )
    elif prob >= 0.50 and days_updated <= 180:
        return (
            "AUDIT_TECHNICAL_LINKS",
            "MEDIUM_PRIORITY",
            "RC_RECENT_DROP: Recent content experiencing traffic drop; audit internal links/cannibalization."
        )
    else:
        return (
            "MONITOR",
            "LOW_PRIORITY",
            "RC_STABLE: Performance metrics within normal range."
        )

playbook_results = df.apply(assign_action_playbook, axis=1)
df["recommended_action"] = [r[0] for r in playbook_results]
df["priority_tier"] = [r[1] for r in playbook_results]
df["reason_code"] = [r[2] for r in playbook_results]

# Sort queue by probability score
action_queue = df[[page_col, "decay_probability", "priority_tier", "recommended_action", "reason_code"] + feature_cols].sort_values(
    by="decay_probability", ascending=False
)

print("=== CONTENT ACTION PLAYBOOK QUEUE GENERATED ===")
print(action_queue.head(10).to_string(index=False))

# ---------------------------------------------------------
# Step C: Export Queue & Figures
# ---------------------------------------------------------
# Save action queue CSV (ignored by git leak-guard as per instructions)
action_queue.to_csv("work/outputs/w07_ranked_action_queue.csv", index=False)

# Plot Action Queue Distribution Figure
plt.figure(figsize=(8, 4.5))
df["recommended_action"].value_counts().plot(kind="bar", color=["#c0392b", "#e67e22", "#2980b9", "#27ae60"])
plt.title("W07 Playbook Action Queue Distribution", fontsize=12, fontweight="bold")
plt.xlabel("Recommended Action")
plt.ylabel("Number of Pages")
plt.xticks(rotation=15)
plt.tight_layout()

figure_path = "work/figures/w07_action_distribution.png"
plt.savefig(figure_path, dpi=300)
plt.close()

print(f"\nAction Queue CSV exported to: work/outputs/w07_ranked_action_queue.csv")
print(f"Distribution Figure committed export to: {figure_path}")

=== CONTENT ACTION PLAYBOOK QUEUE GENERATED ===
          content_id  decay_probability   priority_tier    recommended_action                                                                                     reason_code  content_age_days  days_since_last_update  impressions_90d  ctr  avg_position
content_2d6c50388f53           0.773806   HIGH_PRIORITY       REFRESH_CONTENT                               RC_STALE_DECAY: Content age >365d with high predicted decay risk.               445                       7            14736 0.05          47.4
content_6dd02d2e1d19           0.766635 MEDIUM_PRIORITY     OPTIMIZE_METADATA                       RC_LOW_CTR: Below-average CTR on ranking keywords; needs snippet refresh.               165                     104              278 0.00           2.3
content_3f8c7aed9d6c           0.766635 MEDIUM_PRIORITY     OPTIMIZE_METADATA                       RC_LOW_CTR: Below-average CTR on ranking keywords; needs snippet refresh.               165     

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human Review & Cost/Value Framework
* **Human Review Protocol:** Editorial teams inspect the top 20% highest-risk pages weekly. Pages flagged for `REFRESH_CONTENT` undergo a 15-minute manual check to verify intent freshness before assigning writer resources.
* **Cost/Value ROI Rule:** Prioritize pages with historical `impressions_90d > 10,000`. High-impression pages yield significantly higher ROI upon refresh compared to long-tail low-traffic pages.

---

## 4. Monitoring & Retrain Triggers
What tells us the recommendations have gone stale:
1. **Concept Drift Trigger:** If human review audits identify false positives exceeding 20% over a 2-week window, trigger a probability threshold recalibration.
2. **Data Drift / Macro Algorithmic Shift:** If search engine algorithm updates cause macro shifts in `avg_position` across $\ge 15\%$ of monitored pages, trigger a feature re-baseline and full model retrain.
3. **Scheduled Retrain:** Model parameters are re-evaluated monthly using a rolling 90-day time-aware window.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [2]:
# 5. Exports Verification & Self-Check Execution
print("=== RUNNING SELF-CHECK ===")

# 1. Output files verification
assert os.path.exists("work/outputs/w07_ranked_action_queue.csv"), "Ranked action queue CSV missing from work/outputs!"
assert os.path.exists("work/figures/w07_action_distribution.png"), "Distribution figure missing from work/figures!"

# 2. Content integrity check
exported_df = pd.read_csv("work/outputs/w07_ranked_action_queue.csv")
assert "recommended_action" in exported_df.columns, "Action column missing in exported CSV!"
assert "reason_code" in exported_df.columns, "Reason code column missing in exported CSV!"
assert len(exported_df) > 0, "Exported queue CSV is empty!"

print("Self-check completed successfully! All exports created and verified.")

=== RUNNING SELF-CHECK ===
Self-check completed successfully! All exports created and verified.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
# 5. Exports Verification & Self-Check Execution
import os
import pandas as pd

print("=== RUNNING SELF-CHECK ===")

# 1. Output files verification
assert os.path.exists("work/outputs/w07_ranked_action_queue.csv"), "Ranked action queue CSV missing from work/outputs!"
assert os.path.exists("work/figures/w07_action_distribution.png"), "Distribution figure missing from work/figures!"

# 2. Content integrity check
exported_df = pd.read_csv("work/outputs/w07_ranked_action_queue.csv")
assert "recommended_action" in exported_df.columns, "Action column missing in exported CSV!"
assert "reason_code" in exported_df.columns, "Reason code column missing in exported CSV!"
assert len(exported_df) > 0, "Exported queue CSV is empty!"

print("Self-check completed successfully! All exports created and verified.")

=== RUNNING SELF-CHECK ===
Self-check completed successfully! All exports created and verified.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.